In [ ]:
# Import Libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score

In [ ]:
# Extract ZIP data
import zipfile
with zipfile.ZipFile("/content/drive/MyDrive/Colab_folder/train.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/drive/MyDrive/Colab_folder/extracted_images")

In [ ]:
import os
files_list = os.listdir("/content/drive/MyDrive/Colab_folder/extracted_images/train")

# Count the number of files
num_files = len(files_list)
print(num_files)

25000


In [ ]:
# Organize data into subsets (train, val, test)
import pathlib, shutil

original_dir = pathlib.Path("/content/drive/MyDrive/Colab_folder/extracted_images/train")
new_base_dir = pathlib.Path("/content/drive/MyDrive/Colab_folder/my_images")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir, exist_ok=True)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

# Use more data or less data depending on time and resources (tune these numbers based on available memory)
make_subset("train", start_index=0, end_index=2000)
make_subset("validation", start_index=2000, end_index=2500)
make_subset("test", start_index=2500, end_index=3000)

In [ ]:
# Preprocessing function
labels = ['cat', 'dog']
img_size = 224

def get_data(data_dir):
    data = []
    for label in labels:
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img))[..., ::-1]  # Convert BGR to RGB
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr, class_num])
            except Exception as e:
                print(e)
    return np.array(data, dtype=object)

In [ ]:
# Load and split data
train = get_data('/content/drive/MyDrive/Colab_folder/my_images/train')
val = get_data('/content/drive/MyDrive/Colab_folder/my_images/validation')

# Separate features and labels
x_train, y_train = [], []
x_val, y_val = [], []

for feature, label in train:
    x_train.append(np.array(feature))
    y_train.append(int(label))

for feature, label in val:
    x_val.append(np.array(feature))
    y_val.append(int(label))

# Normalize
x_train = np.array(x_train).astype('float32') / 255
x_val = np.array(x_val).astype('float32') / 255
y_train = np.array(y_train)
y_val = np.array(y_val)

In [ ]:
# Data augmentation for training set
datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train)

In [ ]:
# CNN Architecture (improved)
model = Sequential()
model.add(Conv2D(32, 3, padding="same", activation="relu", input_shape=(224, 224, 3)))
model.add(MaxPool2D())

model.add(Conv2D(64, 3, padding="same", activation="relu"))
model.add(MaxPool2D())

model.add(Conv2D(128, 3, padding="same", activation="relu"))
model.add(MaxPool2D())
model.add(Dropout(0.5))  # Stronger regularization

model.add(Flatten())
model.add(Dense(256, activation="relu"))  # Increased capacity
model.add(Dropout(0.5))
model.add(Dense(1, activation="sigmoid"))  # Binary classification

model.summary()

# Compile model(use low learning rates if not converging)
opt = Adam(learning_rate=0.0001)
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,783,873 (98.36 MB)

 Trainable params: 25,783,873 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Add callbacks for better training control
checkpointer = ModelCheckpoint(filepath='model.weights.best.keras', verbose=1, save_best_only=True)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-6)

# Train the model
hist = model.fit(x_train, y_train, batch_size=32, epochs=20,
                 validation_data=(x_val, y_val),
                 callbacks=[checkpointer, early_stop, lr_reduce],
                 verbose=2, shuffle=True)

Epoch 1/20

Epoch 1: val_loss improved from inf to 0.64364, saving model to model.weights.best.keras
125/125 - 564s - 5s/step - accuracy: 0.5598 - loss: 0.6807 - val_accuracy: 0.5840 - val_loss: 0.6436 - learning_rate: 1.0000e-04
Epoch 2/20

Epoch 2: val_loss improved from 0.64364 to 0.58251, saving model to model.weights.best.keras
125/125 - 531s - 4s/step - accuracy: 0.6670 - loss: 0.6148 - val_accuracy: 0.7060 - val_loss: 0.5825 - learning_rate: 1.0000e-04
Epoch 3/20


KeyboardInterrupt: 

In [ ]:
# Plot training metrics
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
loss = hist.history['loss']
val_loss = hist.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

NameError: name 'hist' is not defined

In [ ]:
# Evaluate on test set
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/Colab_folder/my_images/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 1000 images belonging to 2 classes.


In [ ]:
# Load best model weights
# model.load_weights('model.weights.best.keras')

# Evaluate on validation or test set
# val_loss, val_acc = model.evaluate(x_val, y_val)
# print(f"Restored model accuracy on validation set: {val_acc:.2f}")

# Predict
pred_probs = model.predict(test_generator)

# Convert sigmoid outputs to binary labels (0 or 1)
pred_classes = (pred_probs > 0.5).astype(int).reshape(-1)

# True labels
true_classes = test_generator.classes

# Evaluate
test_accuracy = accuracy_score(true_classes, pred_classes)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


32/32 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step
Test Accuracy: 68.50%
